# NGHIÊN CỨU VÀ TRỰC QUAN HÓA CÁC ĐẶC TRƯNG ÂM THANH TRÊN SPOTIFY

In [52]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

data_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'

df_tracks = pd.read_csv(f'{data_dir}/tracks_dimension.csv')
df_charts = pd.read_parquet(f'{data_dir}/charts_clean.parquet')
df_artist = pd.read_csv(f'{data_dir}/mart_artist_chart.csv')
df_country_year = pd.read_parquet(f'{data_dir}/mart_country_year.parquet')
df_weekly = pd.read_parquet(f'{data_dir}/mart_audio_trend_weekly.parquet')
df_covid = pd.read_csv(f'{data_dir}/covid_weekly_clean.csv')
from pathlib import Path
import sys
PROJECT_ROOT = Path(data_dir).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT / 'dashboard'))
from story_analysis import track_analysis, global_slots
df_weekly = df_weekly[pd.to_datetime(df_weekly['week']).dt.year.between(2017, 2022)].copy()

In [53]:
from pathlib import Path
import pandas as pd
import numpy as np

data_dir_gh = Path("data/processed")
if not (data_dir_gh / "mart_artist_chart.csv").exists():
    data_dir_gh = Path("../data/processed")

csv_path_gh = data_dir_gh / "mart_artist_chart.csv"
df_artist_pop_gh = pd.read_csv(csv_path_gh, dtype={"artist_id": str})
charts_gh = pd.read_parquet(data_dir_gh / "charts_clean.parquet", columns=["artist_id", "track_id", "chart_country"])

global_tracks_gh = charts_gh[charts_gh["chart_country"] == "Global"].groupby("artist_id", observed=True)["track_id"].nunique()
df_artist_pop_gh["n_global_tracks"] = df_artist_pop_gh["artist_id"].map(global_tracks_gh).fillna(0).astype(int)
global_artists_gh = df_artist_pop_gh[df_artist_pop_gh["n_global_tracks"] > 0]

if global_artists_gh.empty:
    raise ValueError("Không có nghệ sĩ xuất hiện trên BXH Global.")

track_cap_gh = int(np.ceil(global_artists_gh["n_global_tracks"].quantile(0.95)))
week_cap_gh = int(np.ceil(global_artists_gh["global_weeks_on_chart"].quantile(0.95)))
log_streams_gh = np.log1p(df_artist_pop_gh["global_streams"])
log_global_gh = np.log1p(global_artists_gh["global_streams"])

df_artist_pop_gh["stream_points"] = 40 * ((log_streams_gh - log_global_gh.min()) / max(log_global_gh.max() - log_global_gh.min(), 1e-12)).clip(0, 1)
df_artist_pop_gh["track_points"] = 25 * (df_artist_pop_gh["n_global_tracks"] / track_cap_gh).clip(0, 1)
df_artist_pop_gh["week_points"] = 20 * (df_artist_pop_gh["global_weeks_on_chart"] / week_cap_gh).clip(0, 1)

rank_gh = df_artist_pop_gh["global_peak_rank"]
df_artist_pop_gh["rank_points"] = np.where(rank_gh.between(1, 200), 15 * (200 - rank_gh) / 199, 0)

point_cols_gh = ["stream_points", "track_points", "week_points", "rank_points"]
df_artist_pop_gh["artist_popularity"] = df_artist_pop_gh[point_cols_gh].sum(axis=1).round(1)
df_artist_pop_gh[point_cols_gh] = df_artist_pop_gh[point_cols_gh].round(2)
df_artist_pop_gh = df_artist_pop_gh.sort_values("artist_popularity", ascending=False, kind="stable").reset_index(drop=True)
df_artist = df_artist_pop_gh.copy()

print(f"Ngưỡng: {track_cap_gh} bài, {week_cap_gh} tuần")
print("Điểm toàn kỳ dùng mô tả nghệ sĩ, chỉ tính trong bộ nhớ; không ghi đè data mart.")
display(df_artist_pop_gh[["artist_name", *point_cols_gh, "artist_popularity"]].head(20))

Ngưỡng: 22 bài, 142 tuần
Điểm toàn kỳ dùng mô tả nghệ sĩ, chỉ tính trong bộ nhớ; không ghi đè data mart.


,artist_name,stream_points,track_points,week_points,rank_points,artist_popularity
0,Bad Bunny,40.00,25.0,20.0,15.00,100.0
1,Drake,39.22,25.0,20.0,15.00,99.2
2,Post Malone,39.04,25.0,20.0,15.00,99.0
3,Ed Sheeran,38.93,25.0,20.0,15.00,98.9
4,J Balvin,38.46,25.0,20.0,15.00,98.5
5,Justin Bieber,38.30,25.0,20.0,15.00,98.3
6,The Weeknd,37.43,25.0,20.0,15.00,97.4
7,Dua Lipa,37.47,25.0,20.0,14.92,97.4
8,Ariana Grande,37.29,25.0,20.0,15.00,97.3
9,Billie Eilish,37.19,25.0,20.0,15.00,97.2


In [54]:
df_analysis = track_analysis(df_charts, df_tracks)
print(f"Mẫu phân tích: {len(df_analysis):,} bài; bỏ 52 tuần đầu và bài chưa đủ cửa sổ tám tuần lịch.")
display(df_analysis[['track_name', 'artist_names', 'first_global_week', 'artist_popularity_prior', 'streams_first_8w']].head(10))

Mẫu phân tích: 4,307 bài; bỏ 52 tuần đầu và bài chưa đủ cửa sổ tám tuần lịch.


,track_name,artist_names,first_global_week,artist_popularity_prior,streams_first_8w
0,Plan A,Paulo Londra,2022-03-24,69.3,107786289
1,Chance,Paulo Londra,2022-04-14,71.5,21082493
2,Una Noche en Medellín,Cris Mj,2022-03-03,0.0,113685371
3,As It Was,Harry Styles,2022-04-07,93.5,451040923
4,Envolver,Anitta,2022-04-14,84.8,126785942
5,Fuera del mercado,Danny Ocean,2022-03-24,55.3,94753351
6,Demasiadas Mujeres,C. Tangana,2022-03-10,37.6,55264425
7,DANCE CRIP,Trueno,2021-11-25,36.7,60777638
8,Tacones Rojos,Sebastian Yatra,2022-02-03,88.8,76044238
9,Jordan,Ryan Castro,2022-03-03,20.9,68579104


## Biểu đồ 1: Hồ sơ 9 đặc trưng âm thanh theo nhóm thứ hạng Spotify Global (2020–2022)

In [55]:
all_features4 = ['danceability', 'energy', 'valence', 'acousticness', 'speechiness', 'instrumentalness', 'liveness', 'loudness', 'tempo']
labels4 = ['Danceability', 'Energy', 'Valence', 'Acousticness', 'Speechiness', 'Instrumentalness', 'Liveness', 'Loudness', 'Tempo']

df_global4 = df_charts[(df_charts['chart_country'] == 'Global') & (df_charts['year'].between(2020, 2022))].copy()
df_global4 = df_global4.drop_duplicates(
    subset=['chart_country', 'week', 'rank', 'track_id'])
band_order4 = [
    'Top 1 đến 10',
    'Hạng 11 đến 50',
    'Hạng 51 đến 100',
    'Hạng 101 đến 200']

df_global4['rank_band'] = pd.cut(
    df_global4['rank'],
    bins=[0, 10, 50, 100, 200],
    labels=band_order4,
    include_lowest=True)

mean_table4 = (df_global4.groupby('rank_band', observed=False)[all_features4].mean().loc[band_order4])

overall_mean4 = df_global4[all_features4].mean()
overall_std4 = df_global4[all_features4].std(ddof=0).replace(0, np.nan)
z_table4 = (mean_table4 - overall_mean4) / overall_std4

text_z4 = [[f'{val:+.2f}' for val in row] for row in z_table4.values]

customdata4 = np.dstack([mean_table4.values, z_table4.values])

max_abs4 = float(np.nanmax(np.abs(z_table4.values)))

fig4 = go.Figure(data=go.Heatmap(z=z_table4.values,
        x=labels4,y=band_order4,
        text=text_z4,texttemplate='%{text}',textfont=dict(size=14),
        customdata=customdata4, colorscale='RdBu_r',
        zmid=0, zmin=-max_abs4, zmax=max_abs4,
        colorbar=dict(
            title='Độ lệch<br>chuẩn z-score',
            thickness=16,
            len=0.85
        ),
        hovertemplate=(
            '<b>Nhóm:</b> %{y}<br>'
            '<b>Đặc trưng:</b> %{x}<br>'
            '<b>Trung bình thực:</b> %{customdata[0]:.3f}<br>'
            '<b>Z-score:</b> %{customdata[1]:+.2f} SD'
            '<extra></extra>'
        )
    )
)

fig4.update_layout(
    title=dict(
        text='<b>Hồ sơ âm thanh theo phân tầng thứ hạng</b>',
        x=0.01,
        font=dict(size=16)
    ),
    xaxis=dict(tickangle=-25, tickfont=dict(size=12)),
    yaxis=dict(autorange='reversed', tickfont=dict(size=12)),
    height=480,
    margin=dict(l=120, r=40, t=75, b=80),
    template='plotly_white'
)
fig4.show()

### Cách tính

$$
z =
\frac{
\text{Trung bình nhóm thứ hạng}
-
\text{Trung bình toàn BXH}
}{
\text{Độ lệch chuẩn toàn BXH}
}
$$
- $z>0$: cao hơn trung bình.
- $z<0$: thấp hơn trung bình.
- $|z|$ càng lớn: khác biệt càng rõ.

### Hiện tượng

- **Top 1–10 nổi bật với sắc đỏ dương tính:** Vượt trội so với mức trung bình toàn bảng xếp hạng ở **Tempo (+0.16 SD)**, **Liveness (+0.14 SD)**, **Danceability (+0.13 SD)**, **Valence (+0.12 SD)** và **Energy (+0.11 SD)**.
- **Sắc xanh âm tính ở nhóm nhạc mộc và không lời:** Top 1–10 thấp hơn trung bình ở **Instrumentalness (-0.12 SD)**, **Acousticness (-0.11 SD)** và **Speechiness (-0.11 SD)**.
- **Loudness gần như bất biến (-0.00 SD):** Màu sắc trung tính thể hiện âm lượng không tạo ra sự khác biệt giữa các phân tầng.

So sánh trực tiếp giữa **Top 1–10** và **Hạng 101–200**:
- **Tempo:** Nhanh hơn **5,53 BPM** (125,78 BPM so với 120,24 BPM).
- **Valence:** Tươi vui hơn **0,035** (0,546 so với 0,511).
- **Danceability:** Dễ nhảy hơn **0,026** (0,704 so với 0,678).
- **Energy:** Sôi động hơn **0,022** (0,647 so với 0,625).
- **Acousticness:** Thấp hơn **0,037** (0,232 so với 0,269).
- **Speechiness:** Thấp hơn **0,012** (0,096 so với 0,108) và **Instrumentalness** thấp hơn **0,010** (0,001 so với 0,011).
- **Loudness:** Chênh lệch thực tế chỉ **0,068 dB** (-6,18 dB so với -6,24 dB).

### Cơ chế giải thích

- **Âm nhạc đại chúng ưu tiên nhịp điệu và năng lượng:** Các ca khúc Top 10 có xu hướng nhanh, bắt tai, tích cực và giàu tính kích thích chuyển động (Danceability, Energy, Tempo cao) nhằm tối ưu cho các ngữ cảnh nghe phổ biến như playlist giải trí, tập luyện, video ngắn và không gian công cộng.
- **Sự thống trị của giọng hát và sản xuất điện tử:** Chỉ số Instrumentalness (-0.12 SD) và Acousticness (-0.11 SD) rất thấp phản ánh xu hướng các bài hit luôn dựa vào giọng hát nổi bật (vocal hooks) và bản phối kỹ thuật số hiện đại.
- **Chỉ số Speechiness (-0.11 SD) thấp:** Không đồng nghĩa với việc Rap kém phổ biến, mà do các bản rap lọt đỉnh bảng xếp hạng thường được melodic hóa (kết hợp giai điệu Pop/R&B dễ hát theo), làm giảm mật độ lời nói thuần túy so với các bản rap underground ở nhóm dưới.

### Kết luận

 Các bài hát Top 10 sở hữu một “dấu vân tay âm học” riêng biệt: nhanh hơn, dễ nhảy hơn, tích cực hơn và mang tính thương mại cao hơn nhóm cuối bảng. Tuy nhiên, mức chênh lệch z-score (+0.11 đến +0.16 SD) là tương đối vừa phải, cho thấy đặc trưng âm thanh tạo lợi thế cạnh tranh nhưng không phải là “công thức độc tôn” quyết định vị trí quán quân.


## Biểu đồ 2: Phân phối đặc trưng âm học giữa các thể loại nhạc

In [56]:
main_genres = ['pop', 'latin', 'k-pop', 'rap', 'dance pop']
df_genre_filtered = df_tracks[df_tracks['artist_genre'].isin(main_genres) & df_tracks['track_id'].isin(df_global4['track_id'])]

fig7 = go.Figure()
box_metrics = ['energy', 'danceability', 'valence', 'tempo']
box_titles = ['Energy', 'Danceability', 'Valence', 'Tempo']

for metric in box_metrics:
    for genre in main_genres:
        data = df_genre_filtered[df_genre_filtered['artist_genre'] == genre][metric]
        fig7.add_trace(go.Box(
            y=data,
            name=genre.upper(),
            visible=(metric == 'energy'),
            boxmean=True
        ))

buttons7 = []
for i, (metric, name) in enumerate(zip(box_metrics, box_titles)):
    vis = [False] * (len(box_metrics) * len(main_genres))
    for j in range(len(main_genres)):
        vis[i * len(main_genres) + j] = True
    buttons7.append(dict(
        label=name,
        method='update',
        args=[{'visible': vis}, {'title': 'Phân phối chỉ số ' + name + ' giữa các thể loại âm nhạc'}]
    ))

fig7.update_layout(
    updatemenus=[dict(
        buttons=buttons7,
        direction='down',
        x=0.01,
        y=1.15,
        showactive=True
    )],
    title='Biểu đồ 2: Phân phối chỉ số Energy giữa các thể loại âm nhạc',
    yaxis_title='Giá trị',
    height=500
)
fig7.show()

### Nhận xét Biểu đồ 
**Hiện tượng:** Biểu đồ tương tác cho thấy bản sắc âm học phân hóa rõ rệt giữa 5 dòng nhạc khi chuyển đổi qua 4 chiều đặc trưng:
- **Energy (Năng lượng):** K-Pop (trung vị 0.73) và Latin (0.72) áp đảo với dải phân vị tập trung ở mức cao; Pop (0.60) và Rap (0.63) có mức năng lượng thấp hơn và độ phân tán rộng hơn.
- **Danceability (Độ dễ nhảy):** Khi chuyển sang Danceability, Latin (0.76) và Rap (0.75) vươn lên dẫn đầu toàn diện với dải hộp co hẹp ở mức rất cao, trong khi Pop (0.65) và K-Pop (0.66) khiêm tốn hơn.
- **Valence (Sắc thái cảm xúc):** Latin độc chiếm vị trí tươi vui nhất (trung vị 0.67, bỏ xa các dòng nhạc khác gần 0.20 đơn vị), thể hiện bản sắc lễ hội đặc trưng; ngược lại Rap (0.44) và Pop (0.45) có phổ cảm xúc thiên về gam màu trầm và u tối hơn.
- **Tempo (Tốc độ nhịp BPM):** Rap dẫn đầu tốc độ với trung vị 126.8 BPM (đặc trưng của beat Trap nhanh); trong khi Latin giữ nhịp đầm chắc nhất quanh 111.0 BPM (nhịp Dembow/Reggaeton).

**Cơ chế giải thích:** 
- Sự khác biệt về Energy và Danceability bắt nguồn từ mục đích biểu diễn: Dance Pop và K-Pop phục vụ trình diễn sân khấu vũ đạo lớn với mật độ âm thanh dày, còn Latin và Rap xây dựng trên nền tảng nhịp groove (Dembow và Hip-hop beat) kích thích chuyển động hình thể tự nhiên.
- Sự phân cực về Valence phản ánh văn hóa âm nhạc: dòng nhạc Latin mang đậm tinh thần tiệc tùng và tích cực, trong khi Rap và Pop chứa đựng nhiều ca từ tự sự, nội tâm và tâm trạng đời sống.

**Kết luận:** Không có thể loại nào "cao hơn toàn diện" trên mọi phương diện; mỗi dòng nhạc đều có một thế mạnh đặc thù (Latin mạnh về Danceability & Valence, K-Pop mạnh về Energy, Rap mạnh về Tempo). Bộ lọc tương tác chứng minh sự cần thiết của việc đánh giá âm nhạc đa chiều thay vì chỉ nhìn vào một chỉ số đơn lẻ.


## Biểu đồ 3: Ma trận tương quan giữa các đặc trưng âm thanh và mức độ phổ biến

In [57]:
corr_features = ["danceability","energy","loudness", "speechiness","acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "artist_popularity_prior",
    "streams_first_8w",]

short_labels = ["danceability","energy","loudness","speechiness","acousticness",
    "instrumentalness", "liveness","valence","tempo","artist_prior","streams_8w",]

df_corr = df_analysis.dropna(subset=corr_features).copy()
raw_corr = df_corr[corr_features].corr(method="spearman").values
n = len(corr_features)

mask = np.triu(np.ones((n, n), dtype=bool), k=0)

z_vals = np.where(mask, np.nan, raw_corr)
text_vals = np.where(mask, "", np.char.mod("%.2f", raw_corr))

fig_corr = go.Figure(
    go.Heatmap(
        z=z_vals,
        x=short_labels,
        y=short_labels,
        zmin=-1,
        zmax=1,
        zmid=0,
        colorscale=[
            [0.0, "#4158D0"],
            [0.25, "#85A5FF"],
            [0.5, "#F7F5F2"],
            [0.75, "#FF9980"],
            [1.0, "#C5221F"],
        ],
        text=text_vals,
        texttemplate="%{text}",
        textfont=dict(size=12, family="Segoe UI", color="#1E293B"),
        showscale=True,
        colorbar=dict(
            thickness=14,
            len=0.9,
            tickvals=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1],
            tickformat=".2f",
            outlinewidth=0,
        ),
        xgap=1.5,
        ygap=1.5,
        hovertemplate="%{y} × %{x}<br>Tương quan: %{z:.2f}<extra></extra>",
    )
)

fig_corr.update_layout(
    title=dict(
        text="Ma trận tương quan — Đặc trưng âm thanh & Popularity",
        font=dict(size=17, family="Segoe UI", color="#1E293B"),
        x=0.5,
        xanchor="center",
    ),
    template="plotly_white",
    width=800,
    height=750,
    font=dict(family="Segoe UI", size=12),
    margin=dict(l=130, r=40, t=70, b=110),
)

fig_corr.update_xaxes(
    tickangle=90,
    tickfont=dict(size=12),
    showgrid=True,
    gridcolor="#ECEFF1",
)
fig_corr.update_yaxes(
    autorange="reversed",
    tickfont=dict(size=12),
    showgrid=True,
    gridcolor="#ECEFF1",
)

fig_corr.show()
corr_matrix = df_corr[corr_features].corr(method='spearman')
display(corr_matrix['streams_first_8w'].drop('streams_first_8w').rename('Spearman ρ với stream tám tuần'))

danceability               0.069325
energy                     0.010585
loudness                   0.092639
speechiness               -0.054692
acousticness              -0.001099
instrumentalness          -0.014027
liveness                  -0.038595
valence                    0.056483
tempo                     -0.016185
artist_popularity_prior    0.119388
Name: Spearman ρ với stream tám tuần, dtype: float64

### Nhận xét:

**Cặp tương quan thuận mạnh nhất:**
- Năng lượng và Âm lượng (đạt dương 0.69): Phản ánh quy chuẩn sản xuất âm nhạc thương mại hiện đại. Những bản nhạc có năng lượng cao, nhịp điệu dồn dập luôn được xử lý và làm chủ với âm lượng lớn nhằm tạo cảm giác kích thích mạnh mẽ trong tai nghe và dàn âm thanh sân khấu.
- Độ dễ nhảy và Cảm xúc tích cực (đạt dương 0.31): Các ca khúc dễ khiêu vũ và bắt nhịp thường mang giai điệu vui tươi, lạc quan, kích thích vận động hình thể tự nhiên.
- Độ dày lời ca và Độ dễ nhảy (đạt dương 0.31): Đặc trưng rõ nét ở các dòng nhạc hiện đại như Rap hay Latin, nơi lời hát liên tục đi cùng tiết tấu nhịp điệu lôi cuốn.

**Cặp tương quan nghịch mạnh nhất:**
- Năng lượng và Độ mộc mạc (đạt âm 0.37): Nhạc cụ mộc tự nhiên đối lập hoàn toàn với cường độ âm thanh điện tử. Một bài hát càng mộc mạc thì năng lượng tổng thể càng dịu êm, thể hiện ranh giới rõ rệt giữa dòng nhạc chữa lành và nhạc khiêu vũ sôi động.
- Âm lượng và Độ mộc mạc (đạt âm 0.27): Nhạc mộc giữ dải động tự nhiên và ít bị đẩy âm lượng tối đa, trái ngược với các dòng nhạc Dance và Pop được sản xuất theo cuộc chiến âm lượng.
**Tác động của âm học và danh tiếng lên hiệu quả thương mại**

- Danh tiếng nghệ sĩ là đòn bẩy quyết định:
  * Hệ số tương quan giữa `artist_prior` (danh tiếng nghệ sĩ tích lũy trước thời điểm ra mắt) và `streams_8w` (lượt stream trong 8 tuần đầu) đạt mức dương đáng kể so với mọi đặc trưng âm học. Điều này chứng minh rằng **tập người hâm mộ sẵn có và vị thế của nghệ sĩ** là yếu tố quyết định tốc độ lan tỏa và quy mô tiếp cận ban đầu của một bài hát mới.
- Đặc trưng âm học đơn lẻ không tự tạo nên hit:
  * Hệ số tương quan tuyến tính/đơn điệu giữa các chỉ số âm thanh riêng lẻ (`Danceability`, `Energy`, `Valence`, v.v.) với `streams_8w` đều ở mức thấp ($|r| < 0.20$).

**Kết luận:** 
- Không tồn tại một "công thức toán học cố định" (như chỉ cần tăng Danceability hay Energy) để đảm bảo bài hát đạt nhiều stream. Đặc trưng âm học đóng vai trò định hình thể loại và trải nghiệm cảm xúc, trong khi danh tiếng nghệ sĩ, chiến lược quảng bá và việc xuất hiện trên các playlist biên tập mới là động lực thúc đẩy lượt nghe thực tế trên bảng xếp hạng toàn cầu.

## Biểu đồ 4: Cơ cấu lượt stream của 80 nghệ sĩ dẫn đầu theo 7 thể loại

In [58]:
genres_tree = ['pop', 'latin', 'k-pop', 'rap', 'dance pop', 'canadian pop', 'trap latino']
credited = df_charts[df_charts['chart_country'].eq('Global') & df_charts['year'].between(2017, 2022)].drop_duplicates(['week', 'rank', 'track_id', 'artist_id'])
artist_streams = credited.groupby('artist_id', observed=True)['streams'].sum().rename('credited_streams')
top_80_artists = df_artist[df_artist['artist_genre'].isin(genres_tree)].drop_duplicates('artist_id').merge(artist_streams, on='artist_id', validate='one_to_one').nlargest(80, 'credited_streams')

# 1. Chuyển đổi sang đơn vị TỶ lượt stream để số ngắn gọn, dễ đọc
top_80_artists['streams_ty'] = top_80_artists['credited_streams'] / 1e9

# 2. Vẽ Treemap với values='streams_ty'
fig9 = px.treemap(
    top_80_artists, 
    path=['artist_genre', 'artist_name'], 
    values='streams_ty', 
    color='artist_genre', 
    color_discrete_map={'pop':'#B3D2EF','latin':'#F0BBCC','k-pop':'#D7C3ED','rap':'#8FA5CB','dance pop':'#FAE7A0','canadian pop':'#BFE3EA','trap latino':'#CE8EA9'}, 
    title='Lượt nghe ghi công cho 80 nghệ sĩ dẫn đầu trên Global (2017–2022)'
)

# 3. Chuẩn hóa nhãn text và hovertemplate
fig9.update_traces(
    textinfo='label+value',
    texttemplate='<b>%{label}</b><br>%{value:.1f} tỷ',
    hovertemplate='<b>%{label}</b><br>Thể loại: %{parent}<br>Stream ghi công: <b>%{value:.2f} tỷ lượt</b><br>Tỷ trọng trong Top 80: <b>%{percentRoot:.1%}</b><extra></extra>'
)
fig9.update_layout(height=550)
fig9.show()
display((top_80_artists.groupby('artist_genre')['credited_streams'].sum() / top_80_artists['credited_streams'].sum() * 100).sort_values(ascending=False).rename('Tỷ trọng trong Top 80 (%)'))


artist_genre
pop             46.125133
trap latino     15.852664
dance pop       15.043329
latin           10.790842
rap              5.621969
canadian pop     4.709425
k-pop            1.856638
Name: Tỷ trọng trong Top 80 (%), dtype: float64

### Nhận xét Biểu đồ

**Hiện tượng:**

- Pop và Latin là hai khối khổng lồ chiếm lĩnh phần lớn biểu đồ: Khối Pop có số lượng nghệ sĩ đông nhất với 24 người. Các ngôi sao dẫn đầu gồm: Drake (16.7 tỷ lượt stream), Ed Sheeran (15.7 tỷ), Justin Bieber (13.6 tỷ), The Weeknd (11.2 tỷ) và Billie Eilish (10.6 tỷ).
- Khối Latin tạo thành đối trọng cực lớn với 19 nghệ sĩ, đóng góp hàng chục tỷ lượt nghe toàn cầu.
Bad Bunny độc chiếm ô cá nhân lớn nhất toàn bộ biểu đồ: Dù nhạc Pop có tổng số nghệ sĩ nhiều hơn, nhưng nghệ sĩ sở hữu ô chữ nhật đơn lẻ to nhất lại là Bad Bunny (đạt tới 19.9 tỷ lượt stream), vượt qua tất cả các siêu sao US-UK. Theo sát phía sau ở khối Latin là J Balvin (14.1 tỷ) và Ozuna (9.6 tỷ).
- Dance Pop và Rap có số lượng nghệ sĩ đông đảo nhưng lượt nghe chia đều hơn: Dance Pop có các tên tuổi nổi bật như Dua Lipa (11.3 tỷ) và Ariana Grande (10.9 tỷ).
- Rap có 21 nghệ sĩ góp mặt nhưng diện tích các ô nhỏ hơn, cho thấy lượt nghe của dòng nhạc Rap được san sẻ cho nhiều nghệ sĩ khác nhau thay vì dồn hết vào một người (dẫn đầu là Cardi B với 6.6 tỷ và DaBaby 5.4 tỷ).
- K-Pop tuy ít nghệ sĩ lọt Top 80 nhưng sở hữu độ tập trung cực cao: Tiêu biểu là nhóm nhạc BTS (đạt tới 8.0 tỷ lượt stream), chứng minh sức mạnh fandom toàn cầu của làn sóng âm nhạc Hàn Quốc.

**Cơ chế giải thích:**

- Sức mạnh của tính phổ quát và khả năng vượt rào cản ngôn ngữ: Nhạc Pop và Latin sở hữu các đặc trưng âm thanh bắt tai, nhịp điệu dễ nhảy và cảm xúc tươi vui. Điều này giúp các bài hát dễ dàng lan tỏa xuyên biên giới, được bật liên tục ở các không gian công cộng, phòng tập và mạng xã hội.
- Hiệu ứng "người thắng cuộc chiếm phần lớn" trên nền tảng số: Thuật toán đề xuất của Spotify và các danh sách phát lớn (Playlist) có xu hướng ưu tiên các ca khúc đang thịnh hành. Điều này tạo ra một vòng lặp: nghệ sĩ càng nổi tiếng thì càng được gợi ý nhiều, dẫn đến việc lượt nghe bị dồn rất mạnh vào một nhóm nhỏ các siêu sao như Bad Bunny, Drake hay Ed Sheeran.

**Kết luận:** 

- Pop và Latin đóng vai trò hai đầu tàu chi phối.
- Thành công của một bài hát trên bảng xếp hạng không chỉ đến từ chất lượng âm thanh thuần túy, mà còn phụ thuộc rất lớn vào thương hiệu cá nhân của nghệ sĩ và độ phủ sóng của dòng nhạc mà họ đại diện

## Biểu đồ 5: Sức mạnh hợp tác: Đơn ca vs. hợp tác biểu diễn

In [59]:
df_tracks_collab = df_analysis[['track_id', 'artists_num', 'streams_first_8w']].copy()
df_tracks_collab['collab_type'] = np.where(df_tracks_collab['artists_num'] > 1, 'Hợp tác (Collab)', 'Đơn ca (Solo)')
track_streams = df_tracks_collab.rename(columns={'streams_first_8w': 'streams'})
counts = track_streams['collab_type'].value_counts()
median_streams = track_streams.groupby('collab_type')['streams'].median() / 1e6
colors = {'Đơn ca (Solo)': '#9FBFE0', 'Hợp tác (Collab)': '#E4A3B3'}
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'xy'}]], subplot_titles=['Số bài trong mẫu Global', 'Stream trung vị · tám tuần'], horizontal_spacing=0.15)
fig.add_trace(go.Pie(labels=counts.index, values=counts.values, hole=0.62, marker=dict(colors=[colors[k] for k in counts.index]), textinfo='label+percent'), row=1, col=1)
bar_types = ['Đơn ca (Solo)', 'Hợp tác (Collab)']
bar_values = [median_streams.get(k, np.nan) for k in bar_types]
fig.add_trace(go.Bar(x=bar_types, y=bar_values, text=[f'{v:.1f} triệu' for v in bar_values], textposition='outside', marker_color=[colors[k] for k in bar_types], hovertemplate='%{x}<br>Stream trung vị tám tuần: %{y:.2f} triệu<extra></extra>'), row=1, col=2)
fig.update_layout(title='Đơn ca và hợp tác trên BXH Global: cùng cửa sổ tám tuần lịch', height=480, template='plotly_white', showlegend=False, margin=dict(l=50,r=40,t=80,b=50))
fig.update_yaxes(title_text='Triệu stream/bài', row=1, col=2)
fig.show()
display(track_streams.groupby('collab_type')['streams'].agg(['count', 'median', 'mean']))

,count,median,mean
collab_type,,,
Hợp tác (Collab),1851,25412152.0,4.328822e+07
Đơn ca (Solo),2456,18884675.5,3.828962e+07


### Nhận xét biểu đồ

- **Cơ cấu bài hát phát hành:** Đơn ca vẫn là hình thức áp đảo tuyệt đối, chiếm 60,8% trong khi các dự án hợp tác từ hai nghệ sĩ trở lên chiếm 39,2%.
- Mặc dù chiếm ít hơn về số lượng, các ca khúc hợp tác ghi nhận mức độ thành công vượt trội với trung bình **53,7 triệu lượt stream trên mỗi bài hát**, cao gấp hơn 3 lần so với mức **17,7 triệu lượt stream** của nhóm Đơn ca.

**Nguyên nhân:**
- Khi hai hoặc nhiều nghệ sĩ kết hợp (đặc biệt là giữa các dòng nhạc khác nhau như Pop kết hợp Latin hoặc Rap), sản phẩm âm nhạc tiếp cận đồng thời nhiều cộng đồng người hâm mộ độc lập, tạo ra lực đẩy ban đầu khổng lồ khi phát hành.
- Thuật toán đề xuất của Spotify ưu tiên đưa các ca khúc hợp tác vào danh sách phát cá nhân hóa của người theo dõi từ cả hai nghệ sĩ, giúp ca khúc dễ dàng bứt phá về lưu lượng và trụ hạng lâu hơn trên bảng xếp hạng.

**Kết luận:**
Đơn ca là hình thức sản xuất truyền thống chiếm phần lớn số lượng trong ngành âm nhạc, nhưng các dự án Hợp tác đóng vai trò đòn bẩy chiến lược tối ưu nhất để bứt phá thị phần và tạo ra các siêu hit toàn cầu.


In [60]:
track_streams = df_tracks_collab.rename(columns={'streams_first_8w': 'streams'}).copy()
track_streams['streams_million'] = track_streams['streams'] / 1_000_000

fig = px.box(
    track_streams,
    x='collab_type',
    y='streams_million',
    color='collab_type',
    color_discrete_map={'Đơn ca (Solo)': '#9FBFE0', 'Hợp tác (Collab)': '#E4A3B3'},
    points=False,
    log_y=True,
    title='Phân phối stream tám tuần trên Global: Đơn ca và Hợp tác (log)' 
)

fig.update_layout(
    xaxis_title=None,
    yaxis_title='Lượt stream (Triệu - Thang Log)',
    height=480,
    template='plotly_white',
    showlegend=False
)

fig.show()

### Chứng minh

- Đường gạch ngang ở giữa hộp (mức phổ biến thông thường) của Hợp tác đạt 2.3 triệu lượt stream, cao hơn gấp 3 lần so với mức 0.7 triệu của Đơn ca.
Vượt trội ở mọi vị trí:

- Ngay cả đáy hộp (nhóm bài thấp) hay đỉnh hộp (nhóm bài cao) của Hợp tác đều nằm cao hơn Đơn ca một khoảng cách rất rõ ràng.
Kết luận:

- Biểu đồ hộp chứng minh rằng: Các bài hát Hợp tác thực sự đạt lượt nghe cao hơn một cách toàn diện, hoàn toàn không phải do bị kéo lệch ảo bởi một vài bài siêu hit cá biệt.

## Biểu đồ 6: Sự co cụm thời lượng bài hát: Hiện tượng TikTok-hóa âm nhạc (2017 – 2022)

In [61]:
df_glob_dur = df_weekly[df_weekly['chart_country'] == 'Global'].copy()
df_glob_dur['week'] = pd.to_datetime(df_glob_dur['week'])
df_glob_dur = df_glob_dur.sort_values('week')
df_glob_dur['duration_sec'] = df_glob_dur['duration_weighted'] / 1000
df_glob_dur['duration_rolling'] = df_glob_dur['duration_sec'].rolling(4, min_periods=1).mean()

fig_dur = go.Figure()

fig_dur.add_trace(go.Scatter(
    x=df_glob_dur['week'],
    y=df_glob_dur['duration_sec'],
    mode='lines',
    name='Thời lượng hàng tuần',
    line=dict(color='#C7CFD5', width=1.5),
    hovertemplate='Ngày: %{x|%d/%m/%Y}<br>Thời lượng: %{y:.1f}s<extra></extra>'
))

fig_dur.add_trace(go.Scatter(
    x=df_glob_dur['week'],
    y=df_glob_dur['duration_rolling'],
    mode='lines',
    name='Xu hướng (Trung bình 4 tuần)',
    line=dict(color='#4E708A', width=3),
    hovertemplate='Ngày: %{x|%d/%m/%Y}<br>Xu hướng: %{y:.1f}s<extra></extra>'
))

fig_dur.update_layout(
    title='<b>Thời lượng bài hát trên BXH Global (2017 – 2022)</b>',
    xaxis_title='Thời gian',
    yaxis_title='Thời lượng trung bình (giây)',
    height=500,
    template='plotly_white',
    hovermode='x unified',
    margin=dict(l=60, r=40, t=75, b=50),
    legend=dict(orientation='h', yanchor='top', y=-0.12, xanchor='center', x=0.5)
)

fig_dur.show()

duration_yearly = df_glob_dur.groupby(df_glob_dur['week'].dt.year)['duration_sec'].agg(['mean', 'count'])
display(duration_yearly.rename(columns={'mean':'Thời lượng trung bình (giây)', 'count':'Số tuần quan sát'}))

,Thời lượng trung bình (giây),Số tuần quan sát
week,,
2017,217.960230,52
2018,209.157769,52
2019,201.189301,52
2020,198.495772,53
2021,200.122580,52
2022,205.705048,28


### Nhận xét biểu đồ

**Xu hướng sụt giảm thời lượng liên tục và rõ rệt:**
- Giai đoạn 2017: Đường xu hướng thời lượng trung bình neo ở mức cao quanh **218 giây (tương đương 3 phút 38 giây)**.
- Quỹ đạo tụt dốc liên tục (2018 – 2020): Thời lượng trung bình dốc thẳng đứng đi xuống qua từng năm: giảm xuống ~209 giây (2018), ~201 giây (2019) và chạm đáy quanh **198 giây (3 phút 18 giây)** vào cuối năm 2020. Chỉ sau 3 năm, một ca khúc hit trung bình đã **bị cắt ngắn đi đúng 20 giây**.
- Giai đoạn 2021 – 2022: Thời lượng bài hát dao động ổn định quanh ngưỡng ngắn **200 – 205 giây (~3 phút 20 giây)**, hoàn toàn không có dấu hiệu phục hồi về mức dài trên 3 phút 35 giây như thời kỳ 2017.
**Cơ chế giải thích:**

**Nguyên nhân**

 - Spotify tính tiền bản quyền cho 1 lượt stream hợp lệ khi người dùng nghe qua giây thứ 30 (bất kể bài hát dài 2 phút hay 5 phút). Một ca khúc dài 2 phút 30 giây có thể được phát lại tới 24 lần trong 1 giờ, trong khi bài hát dài 4 phút chỉ phát được 15 lần. Do đó, các hãng đĩa và nhà sản xuất chủ động rút ngắn thời lượng bài hát nhằm tối ưu hóa doanh thu từ lượt nghe lặp lại.
- Sự bùng nổ của TikTok từ năm 2019 – 2020 đã tái định hình thói quen tiếp nhận thông tin của thế hệ người nghe mới. Người dùng quen với các đoạn âm thanh cao trào ngắn gọn từ 15 đến 30 giây, dẫn đến sự suy giảm khả năng kiên nhẫn khi nghe nhạc dài. Các nhà sản xuất âm nhạc buộc phải: Cắt bỏ phần dạo đầu (intro dài), vào thẳng phần hát hoặc điệp khúc ngay từ những giây đầu tiên. Lược bỏ hoặc thu ngắn đoạn chuyển tiếp (bridge) và phần kết thúc (outro).

**Kết luận:**

- Định dạng bài hát dưới 3 phút không còn là ngoại lệ mà đã trở thành công thức bắt buộc để các ca khúc thích ứng với thuật toán gợi ý và văn hóa tiêu thụ nhanh của kỷ nguyên số.
- Chiến lược phát hành: Các nghệ sĩ ưu tiên ra mắt các bản nhạc cô đọng, nhịp điệu nhanh và bắt tai ngay lập tức để giữ chân thính giả không bấm nút bỏ qua (skip), bảo đảm thứ hạng cao trên các bảng xếp hạng streaming toàn cầu.


## Biểu đồ 7: Bản đồ phân bổ âm học toàn cầu

In [62]:
df_map = df_country_year[(df_country_year['year'] == 2021) & (df_country_year['chart_country'] != 'Global')].dropna(subset=['country_code'])

map_metrics = [
    ('valence_weighted', 'Valence: trầm buồn sang tươi vui', 'Tealrose'),
    ('danceability_weighted', 'Danceability: độ bắt nhịp khiêu vũ', 'Viridis'),
    ('acousticness_weighted', 'Acousticness: độ mộc mạc nhạc cụ', 'YlOrBr'),
    ('energy_weighted', 'Energy: mức độ sôi động và cường độ', 'Plasma')
]

fig_map = go.Figure()

for i, (col, label, colorscale) in enumerate(map_metrics):
    fig_map.add_trace(go.Choropleth(
        locations=df_map['country_code'],
        z=df_map[col],
        text=df_map['chart_country'],
        name=label.split(':')[0],
        colorscale=colorscale,
        colorbar_title=col.replace('_weighted', '').capitalize(),
        visible=(i == 0),
        hovertemplate='<b>%{text}</b><br>' + col.replace('_weighted', '').capitalize() + ': %{z:.3f}<extra></extra>'
    ))

buttons_map = []
for i, (col, label, colorscale) in enumerate(map_metrics):
    vis = [False] * len(map_metrics)
    vis[i] = True
    buttons_map.append(dict(
        label=label.split(':')[0].strip(),
        method='update',
        args=[
            {'visible': vis},
            {'title': f'Bản đồ phân bổ âm học toàn cầu: {label}'}
        ]
    ))

fig_map.update_layout(
    title='Bản đồ phân bổ âm học toàn cầu: ' + map_metrics[0][1],
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    updatemenus=[dict(
        buttons=buttons_map,
        direction='down',
        x=0.01,
        y=1.15,
        showactive=True
    )],
    height=600,
    margin=dict(l=0, r=0, t=75, b=20)
)
fig_map.show()


### Nhận xét biểu đồ

- **Valence (Sắc thái cảm xúc):** Xuất hiện sự phân cực Đông – Tây rõ nét nhất toàn cầu. Khu vực Mỹ Latinh (Brazil, Colombia, Mexico) rực sáng với gam màu tươi vui cao nhất thế giới (Valence đạt từ 0.65 đến 0.69). Ngược lại, khu vực Đông Á và Đông Nam Á (Đài Loan, Hàn Quốc, Việt Nam với 0.466) mang sắc màu trầm lắng rõ rệt (Valence đều dưới 0.48).
- **Acousticness (Độ mộc mạc):** Là chỉ số có độ tương phản vùng miền mạnh nhất. Các quốc gia Đông Nam Á (Indonesia 0.476, Việt Nam 0.465, Philippines) nhuộm màu đậm nhất thế giới, thể hiện tỷ lệ nhạc cụ mộc cao áp đảo. Đối lập hoàn toàn, Nhật Bản (0.173) và các nước Bắc Âu có sắc màu nhạt nhất do tỷ lệ phối khí điện tử chiếm ưu thế.
- **Danceability (Độ dễ khiêu vũ):** Toàn bộ khối Nam Mỹ (dẫn đầu là Chile 0.752) và vùng Caribe áp đảo toàn diện về khả năng kích thích vận động, trong khi các thị trường châu Á (Nhật Bản, Indonesia 0.588) duy trì độ bắt nhịp ở mức vừa phải.
- **Energy (Mức độ sôi động):** Phản ánh cường độ âm thanh phân hóa, nơi Nhật Bản vươn lên nhóm dẫn đầu (0.727) đối lập với tính êm dịu, trữ tình tại Đông Nam Á (Indonesia 0.514, Việt Nam 0.532).

**Nguyên nhân:**
- **Bản sắc văn hóa và thói quen thưởng thức:** Sự thống trị của Valence và Danceability tại Mỹ Latinh phản ánh văn hóa lễ hội bản địa gắn liền với các thể loại nhạc khiêu vũ truyền thống và hiện đại (Salsa, Samba, Reggaeton). Tại Đông Á và Đông Nam Á, thị hiếu đại chúng chuộng các dòng Pop Ballad trữ tình, tự sự, đòi hỏi không gian âm thanh lắng đọng, nhiều tiếng đàn mộc (Guitar, Piano) và giai điệu sâu lắng.
- **Đặc trưng ngành sản xuất âm nhạc nội địa:** Thị trường Nhật Bản có nền công nghiệp J-Pop, J-Rock và nhạc hoạt hình (Anime OST) phát triển mạnh với phong cách hòa âm điện tử phức hợp, nhịp trống dồn dập, đẩy chỉ số Energy lên cao và kéo Acousticness xuống mức thấp kỷ lục.


## Biểu đồ 8: Dấu vân tay ADN âm nhạc: Việt Nam vs. Nhật Bản

In [63]:
df_2021 = df_country_year[df_country_year['year'] == 2021].set_index('chart_country')

features = ['acousticness_weighted', 'danceability_weighted', 'energy_weighted', 'valence_weighted', 'speechiness_weighted', 'liveness_weighted']
categories = ['<b>Acousticness</b>', '<b>Danceability</b>', '<b>Energy</b>', '<b>Valence</b>', '<b>Speechiness</b>', '<b>Liveness</b>']
cat_closed = categories + [categories[0]]

v_vn = df_2021.loc['Vietnam', features].tolist() + [df_2021.loc['Vietnam', features[0]]]
v_jp = df_2021.loc['Japan', features].tolist() + [df_2021.loc['Japan', features[0]]]

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=v_vn,
    theta=cat_closed,
    fill='toself',
    name='Vietnam',
    fillcolor='rgba(231, 76, 60, 0.35)',
    line=dict(color='#E74C3C', width=2.5),
    hovertemplate='%{theta}: %{r:.3f}<extra></extra>'
))

fig.add_trace(go.Scatterpolar(
    r=v_jp,
    theta=cat_closed,
    fill='toself',
    name='Japan',
    fillcolor='rgba(52, 152, 219, 0.35)',
    line=dict(color='#3498DB', width=2.5),
    hovertemplate='%{theta}: %{r:.3f}<extra></extra>'
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 0.85], tickfont=dict(size=10)),
        angularaxis=dict(tickfont=dict(size=12))
    ),
    title=dict(text='<b>Dấu vân tay ADN âm nhạc: Việt Nam vs. Nhật Bản (2021, có trọng số stream)</b>', x=0.01, y=0.98),
    height=560,
    template='plotly_white',
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.12,
        xanchor='center',
        x=0.5,
        font=dict(size=12)
    ),
    margin=dict(l=60, r=60, t=60, b=70)
)

fig.show()
display(df_2021.loc[['Vietnam','Japan'], features].T)

chart_country,Vietnam,Japan
acousticness_weighted,0.464511,0.172515
danceability_weighted,0.668318,0.609756
energy_weighted,0.532463,0.727374
valence_weighted,0.466393,0.591120
speechiness_weighted,0.068224,0.063369
liveness_weighted,0.161919,0.200678


### Nhận xét:

- Acousticness (Độ mộc mạc): Việt Nam đạt mức $0.47$, gấp gần $2.8$ lần so với Nhật Bản ($0.17$). Đây là một trong những chỉ số mộc mạc cao nhất trên bản đồ streaming toàn cầu.
- Energy (Cường độ & Năng lượng): Nhật Bản đạt mức $0.73$, vượt trội hoàn toàn so với mức $0.53$ của Việt Nam. Nhật Bản nằm trong nhóm thị trường có cường độ âm thanh sôi động và dồn dập hàng đầu thế giới.
- Valence (Sắc thái cảm xúc): Nhật Bản đạt $0.59$, mang âm hưởng tươi sáng và lạc quan hơn; trong khi Việt Nam chỉ đạt $0.46$, thể hiện xu hướng ưa chuộng các giai điệu trầm buồn, sâu lắng hoặc mang tâm trạng u hoài.
- Danceability (Độ bắt nhịp khiêu vũ): Cả hai thị trường có mức độ tương đồng tương đối cao, dao động từ $0.61$ (Nhật Bản) đến $0.66$ (Việt Nam).
- Speechiness & Liveness: Cả hai quốc gia đều duy trì ở mức thấp (Speechiness: 0.06 – 0.07; Liveness: 0.16 – 0.20), cho thấy thị hiếu chung của cả hai nước vẫn là các bản thu âm hoàn chỉnh trong studio (Studio Production) và thiên về giai điệu hát (Melodic Vocal) thay vì nhạc Rap thuần túy hay thu âm biểu diễn trực tiếp (Live).

**Phân tích nguyên nhân & Bản sắc văn hóa âm nhạc**

- Thị trường Việt Nam Đế chế của Pop Ballad và nhạc tự sự: Chỉ số Acousticness cao vượt trội kết hợp với Valence thấp ($0.46$) phản ánh thị hiếu lâu năm của khán giả đại chúng Việt Nam dành cho thể loại Pop Ballad, R&B tự sự và Indie acoustic. Các bài hát thịnh hành thường có phần đệm đàn piano/guitar mộc mạc, tốc độ vừa phải, giọng hát dẫn dắt cảm xúc và ca từ sâu lắng nói về những nỗi buồn trong tình yêu.
- Thị trường Nhật Bản với văn hóa J-Pop, J-Rock và Anime OST: Bảng xếp hạng thị trường này bị chi phối mạnh mẽ bởi các ca khúc nhạc phim Anime, J-Rock và J-Pop điện tử. Đây là những thể loại có cấu trúc phối khí đa tầng, mật độ nhạc cụ điện tử và trống dồn dập, nhịp điệu nhanh và tiết tấu tràn đầy sinh lực.

**Ý nghĩa phân tích**
- Xóa bỏ định kiến gộp nhóm địa lý: Biểu đồ chứng minh rằng yếu tố vị trí địa lý hay văn hóa phương Đông không đồng nghĩa với việc có chung thị hiếu tiêu dùng âm nhạc. Mỗi quốc gia sở hữu một "dấu vân tay ADN" âm học riêng biệt.
- Định hướng chiến lược phát hành: Đối với các nghệ sĩ và hãng đĩa muốn mở rộng thị trường, sản phẩm phát hành tại Việt Nam cần chú trọng đến chiều sâu ca từ và giai điệu mộc mạc, trong khi sản phẩm thâm nhập thị trường Nhật Bản bắt buộc phải đầu tư vào năng lượng phối khí và tiết tấu dồn dập.

**Giới hạn nghiên cứu**

- Dữ liệu này được tính trên Top 200 Spotify Charts, đại diện cho nhóm người dùng trẻ đô thị sử dụng ứng dụng streaming trực tuyến tại Việt Nam và Nhật Bản, chưa đại diện cho toàn bộ 100% thị hiếu âm nhạc dân gian/đại chúng truyền thống của mọi lứa tuổi ở hai quốc gia.

## Biểu đồ 9: Tính mùa vụ trong âm nhạc: Danceability vs acousticness theo 12 tháng

In [64]:
df_glob = df_weekly[df_weekly['chart_country'] == 'Global'].copy()
df_glob['month'] = pd.to_datetime(df_glob['week']).dt.month
monthly_avg = df_glob.groupby('month')[['danceability_weighted', 'acousticness_weighted']].mean().reset_index()

fig10 = px.bar(
    monthly_avg,
    x='month',
    y=['danceability_weighted', 'acousticness_weighted'],
    barmode='group',
    title='Tính mùa vụ trong âm nhạc: Danceability vs acousticness theo 12 tháng',
    labels={'month': 'Tháng trong năm', 'value': 'Giá trị trung bình', 'variable': 'Đặc trưng âm học'},
    color_discrete_map={'danceability_weighted': '#1DB954', 'acousticness_weighted': '#E67E22'}
)
fig10.update_layout(
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    height=500
)
fig10.show()


### Nhận xét biểu đồ

Chỉ số Danceability duy trì ổn định quanh mức 0.70 từ tháng 1 đến tháng 11, nhưng suy giảm rõ rệt vào tháng 12. Ngược lại, Acousticness duy trì ở mức thấp khoảng 0.22 đến 0.24 trong suốt 11 tháng đầu năm, sau đó tăng vọt lên mức cao nhất vào tháng 12.

**Nguyên nhân:** Hiện tượng biến thiên đột ngột vào tháng 12 là hệ quả trực tiếp của Giáng sinh. Trong giai đoạn này, các bài hát kinh điển truyền thống từ thế kỷ trước vốn sử dụng nhạc cụ mộc, dàn dây và ít phách điện tử tái xuất hiện và chiếm lĩnh các vị trí dẫn đầu bảng xếp hạng, làm thay đổi cấu trúc âm học bình quân của toàn bộ thị trường.

**Kết luận:** Âm nhạc trực tuyến sở hữu tính mùa vụ chu kỳ rõ rệt hàng năm, trong đó tháng 12 là giai đoạn dị biệt duy nhất khi người nghe tạm rời xa nhạc nhảy điện tử để tìm về âm hưởng mộc mạc hoài niệm.


## Biểu đồ 10: Tương quan [Chỉ số] và số ca tử vong COVID-19

In [65]:
df_covid_proc = df_covid.copy()
df_weekly_proc = df_weekly.copy()
df_covid_proc['week'] = pd.to_datetime(df_covid_proc['week'])
df_weekly_proc['week'] = pd.to_datetime(df_weekly_proc['week'])

cov_glob = df_covid_proc[df_covid_proc['country'] == 'Global']
mus_glob = df_weekly_proc[df_weekly_proc['chart_country'] == 'Global']

df_merged = mus_glob.merge(cov_glob, on='week').sort_values('week')
df_merged = df_merged[df_merged['week'].between('2020-01-01', '2022-07-31')]

features_9 = [
    ('acousticness_weighted', 'Acousticness', '#9B59B6', '0.0 - 1.0'),
    ('energy_weighted', 'Energy', '#E67E22', '0.0 - 1.0'),
    ('valence_weighted', 'Valence', '#2ECC71', '0.0 - 1.0'),
    ('danceability_weighted', 'Danceability', '#1DB954', '0.0 - 1.0'),
    ('loudness_weighted', 'Loudness', '#34495E', 'dB'),
    ('speechiness_weighted', 'Speechiness', '#3498DB', '0.0 - 1.0'),
    ('instrumentalness_weighted', 'Instrumentalness', '#E74C3C', '0.0 - 1.0'),
    ('tempo_weighted', 'Tempo', '#16A085', 'BPM'),
    ('liveness_weighted', 'Liveness', '#F39C12', '0.0 - 1.0')
]

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(
        x=df_merged['week'],
        y=df_merged['weekly_deaths'],
        name='Tử vong COVID-19/tuần',
        marker=dict(color='rgba(231, 76, 60, 0.35)'),
        hovertemplate='Tử vong: %{y:,.0f} ca<extra></extra>'
    ),
    secondary_y=True
)

for i, (col, name, color, unit) in enumerate(features_9):
    fig.add_trace(
        go.Scatter(
            x=df_merged['week'],
            y=df_merged[col],
            name=name,
            mode='lines',
            line=dict(color=color, width=2.5),
            visible=(i == 0),
            hovertemplate=name + ': %{y:.3f}<extra></extra>'
        ),
        secondary_y=False
    )

buttons = []
for i, (col, name, color, unit) in enumerate(features_9):
    vis = [True] + [False] * len(features_9)
    vis[i + 1] = True
    buttons.append(dict(
        label=name,
        method='update',
        args=[
            {'visible': vis},
            {
                'title': f'<b>Biểu đồ 9: Tương quan {name} và số ca tử vong COVID-19</b>',
                'yaxis.title.text': f'Chỉ số {name} ({unit})'
            }
        ]
    ))

fig.update_layout(
    title=f'<b>Tương quan {features_9[0][1]} và số ca tử vong COVID-19</b>',
    height=520,
    template='plotly_white',
    hovermode='x unified',
    margin=dict(l=60, r=60, t=80, b=50),
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=0.01,
        y=1.18,
        showactive=True
    )]
)

fig.update_yaxes(title_text=f'Chỉ số {features_9[0][1]} ({features_9[0][3]})', secondary_y=False)
fig.update_yaxes(title_text='Ca tử vong / tuần', secondary_y=True, showgrid=False)

fig.show()

covid_pairs = df_merged[['week', 'weekly_deaths', 'acousticness_weighted']].dropna().copy()
raw_r = covid_pairs['weekly_deaths'].corr(covid_pairs['acousticness_weighted'])
centered = covid_pairs[['weekly_deaths', 'acousticness_weighted']] - covid_pairs.groupby(covid_pairs['week'].dt.month)[['weekly_deaths', 'acousticness_weighted']].transform('mean')
adjusted_r = centered['weekly_deaths'].corr(centered['acousticness_weighted'])
print(f"Pearson r: gốc = {raw_r:.3f}; sau trừ trung bình tháng lịch = {adjusted_r:.3f}; {len(covid_pairs)} tuần.")

Pearson r: gốc = 0.345; sau trừ trung bình tháng lịch = 0.315; 130 tuần.


### Nhận xét biểu đồ

- **Số ca tử vong COVID-19:**
  - Khởi đầu từ 0 vào tháng 01/2020, nhanh chóng vọt lên đỉnh sóng đầu tiên vào tháng 04/2020 (~50.000 ca/tuần).
  - Đạt **đỉnh cao nhất toàn biểu đồ vào tháng 01/2021 với hơn 102.000 ca tử vong/tuần**.
  - Tiếp tục duy trì đỉnh thứ hai vào giữa năm 2021 (~90.000 ca/tuần), đỉnh thứ ba đầu năm 2022 (~75.000 ca/tuần), sau đó dốc thẳng xuống mức thấp vào giữa năm 2022.

- **9 đặc trưng âm học:**
  1. **Acousticness:** Là đường biến động mạnh mẽ nhất; ngay khi đường đỏ tử vong dựng đứng từ tháng 03/2020, đường tím bật tăng mạnh từ mức 0.22 lên đỉnh cao 0.36, liên tục neo ở mức cao trong suốt các giai đoạn đỉnh dịch năm 2020–2021.
  2. **Energy:** Chạy dốc ngược hướng với đường tử vong; chạm đáy thấp nhất 0.58 đúng vào giai đoạn dịch bùng phát cuối năm 2020 và chỉ bật tăng trở lại đỉnh 0.67 khi đường đỏ tử vong hạ nhiệt vào giữa năm 2022.
  3. **Valence:** Chạy ổn định trong dải 0.50 đến 0.58; ngay cả tại tuần đỉnh điểm tử vong kỷ lục (tháng 01/2021), đường xanh lá vẫn đi ngang trên mốc 0.50, hoàn toàn không có cú rơi tự do nào xuống đáy tiêu cực.
  4. **Danceability:** Dao động trong dải 0.62 đến 0.72, có xu hướng võng xuống tại các mốc thời gian đường tử vong lập đỉnh.
  5. **Loudness:** Chạy ở dải âm từ -5.7 dB đến -7.5 dB, võng sâu nhất (âm hơn, tức âm thanh nhỏ hơn) vào đợt bùng dịch cuối năm 2020.
  6. **Speechiness:** Chạy ở dải thấp quanh 0.07 đến 0.13, có xu hướng giảm nhẹ dần về cuối chu kỳ đại dịch.
  7. **Instrumentalness:** Nằm sát đáy trục hoành ở mức cực thấp (0.004 đến 0.024), hầu như không vượt lên được.
  8. **Tempo:** Chạy dạng sóng răng cưa quanh ngưỡng 117 đến 125 BPM, giữ nhịp tương đối đều qua các năm.
  9. **Liveness:** Chạy gần như một đường thẳng nằm ngang phẳng lì ở dải hẹp 0.15 đến 0.20, không phản ứng trước các đợt tăng của đường đỏ tử vong.

**Cơ chế giải thích:**
- **Sự đối lập giữa Acousticness và Energy:** Phản ánh trực tiếp tác động của các lệnh phong tỏa. Khi số ca tử vong tăng cao khiến các tụ điểm giải trí và câu lạc bộ đóng cửa, công chúng chuyển sang sinh hoạt tại gia, thúc đẩy nhu cầu nghe các dòng nhạc mộc mạc, nhẹ nhàng (đường tím vọt lên) và giảm nhu cầu nhạc điện tử sôi động, âm lượng lớn (đường cam và đường xám võng xuống).
- **Valence kiên trì đi ngang trên mốc 0.50:** Cho thấy người nghe không dùng âm nhạc để gặm nhấm sự bi quan trong lúc số ca tử vong đạt đỉnh, mà chủ động chọn bài hát có giai điệu ấm áp để tự xoa dịu tâm lý và duy trì niềm hy vọng.
- **Đường Tempo và Liveness phẳng lì:** Cho thấy dịch bệnh thay đổi phong cách thưởng thức (mộc hơn, êm hơn) chứ không làm thay đổi các quy chuẩn nhịp phách cơ học và chuẩn thu âm phòng thu.

**Kết luận:**
Biểu đồ trực quan hóa sắc nét thông điệp: Các đợt bùng phát tử vong COVID-19 kéo theo sự trỗi dậy của âm nhạc mộc mạc (`Acousticness` tăng) và sự thu hẹp của âm nhạc cường độ mạnh (`Energy` giảm), nhưng đường cảm xúc (`Valence`) kiên định không sụt giảm — minh chứng thị giác hoàn hảo cho kết luận **"Êm hơn nhưng không buồn hơn"**.

